In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"  # limit sklearn internal threading

import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA




In [2]:
BASE_PATH = Path("/kaggle/input/plant-pathology-2020-fgvc7")
TRAIN_CSV = BASE_PATH / "train.csv"
TEST_CSV = BASE_PATH / "test.csv"
SAMPLE_SUB = BASE_PATH / "sample_submission.csv"
IMG_DIR = BASE_PATH / "images"




In [3]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB)

LABEL_COLS = ["healthy", "multiple_diseases", "rust", "scab"]
y = train_df[LABEL_COLS].values.astype(np.float32)




In [4]:
from concurrent.futures import ThreadPoolExecutor
import os


def extract_features(image_path, size=(64, 64)):
    """
    Load an image, resize, and return a feature vector composed of:
      - mean RGB (3,)
      - std RGB   (3,)
      - per‑channel colour histograms (16 bins each → 48,)
      - edge‑strength statistic (1,)
      - flattened grayscale (size[0] * size[1],)
    Reducing the resize target to 64×64 lowers dimensionality and noise.
    """
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img = img.resize(size)
        arr = np.asarray(img, dtype=np.float32) / 255.0  # (h, w, 3)

        mean_rgb = arr.mean(axis=(0, 1))
        std_rgb = arr.std(axis=(0, 1))

        hist_bins = 16
        hist_features = np.concatenate(
            [
                np.histogram(
                    arr[:, :, ch], bins=hist_bins, range=(0.0, 1.0), density=True
                )[0]
                for ch in range(3)
            ]
        )

        gray = np.dot(arr, [0.2989, 0.5870, 0.1140])
        gy, gx = np.gradient(gray)
        grad_mag = np.sqrt(gx**2 + gy**2)
        edge_mean = np.array([grad_mag.mean()], dtype=np.float32)

        gray_flat = gray.ravel()
        return np.concatenate([mean_rgb, std_rgb, hist_features, edge_mean, gray_flat])


def build_feature_matrix(df):
    """
    Return a (n_samples, feature_dim) array of image features.
    Parallel I/O is limited to a reasonable number of workers to avoid
    excessive context switching while still benefitting from concurrency.
    """
    img_paths = [IMG_DIR / f"{img_id}.jpg" for img_id in df["image_id"]]

    max_workers = min(8, os.cpu_count() or 1)  # limit workers for efficiency
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        feats_list = list(executor.map(extract_features, img_paths))

    feats = np.stack(feats_list).astype(np.float32)
    return feats




In [5]:
X_train = build_feature_matrix(train_df)
X_test = build_feature_matrix(test_df)




In [6]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y, test_size=0.2, random_state=42, stratify=y.argmax(axis=1)
)

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)

# Keep slightly fewer PCA components to reduce noise (99.8% variance)
pca = PCA(n_components=0.98, random_state=42)
X_tr_pca = pca.fit_transform(X_tr_scaled)
X_val_pca = pca.transform(X_val_scaled)

# More regularized GradientBoosting: more trees, smaller learning rate, shallower depth
base_clf = GradientBoostingClassifier(
    n_estimators=1200,  # increase trees for better fit
    learning_rate=0.05,  # smaller step size
    max_depth=3,  # shallower trees to reduce over‑fitting
    max_features="sqrt",
    subsample=0.9,
    random_state=42,
)
clf = OneVsRestClassifier(base_clf)
clf.fit(X_tr_pca, y_tr)

val_preds = clf.predict_proba(X_val_pca)
val_auc = np.mean(
    [roc_auc_score(y_val[:, i], val_preds[:, i]) for i in range(len(LABEL_COLS))]
)
print(f"Validation mean ROC‑AUC: {val_auc:.4f}")




Validation mean ROC‑AUC: 0.5449


In [7]:
X_train_scaled = scaler.fit_transform(X_train)
X_train_pca = pca.fit_transform(X_train_scaled)

clf.fit(X_train_pca, y)




OneVsRestClassifier(estimator=GradientBoostingClassifier(learning_rate=0.05,
                                                         max_features='sqrt',
                                                         n_estimators=1200,
                                                         random_state=42,
                                                         subsample=0.9))

In [8]:
X_test_scaled = scaler.transform(X_test)
X_test_pca = pca.transform(X_test_scaled)

test_preds = clf.predict_proba(X_test_pca)  # shape (n_test, 4)




In [9]:
submission = pd.DataFrame(test_preds, columns=LABEL_COLS)
submission.insert(0, "image_id", test_df["image_id"])
submission.to_csv("submission.csv", index=False, float_format="%.6f")
print("Submission file written to submission.csv")

Submission file written to submission.csv
